# Replay smolvla vs two pi05 checkpoints on their training data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alimertturker/lerobot-policy-probe/blob/main/examples/replay_colab.ipynb)

This notebook compares what three checkpoints would command, joint by joint, against the human demonstrations in the dataset they were both trained on:

| policy | checkpoint |
|---|---|
| smolvla | [`alimerido/smolvla-wrist-top-cube-v3`](https://huggingface.co/alimerido/smolvla-wrist-top-cube-v3) |
| pi05 (3k steps) | [`alimerido/pi05-wrist-top-generic-3k-tn`](https://huggingface.co/alimerido/pi05-wrist-top-generic-3k-tn) |
| pi05 (10k steps) | [`alimerido/pi05-wrist-top-cube-10k`](https://huggingface.co/alimerido/pi05-wrist-top-cube-10k) |

Dataset: [`alimerido/wrist-top-cube_20260705_134536`](https://huggingface.co/datasets/alimerido/wrist-top-cube_20260705_134536), which all three `train_config.json` files name as their training set.

**Runtime:** go to **Runtime → Change runtime type** and pick an **L4** or **A100** GPU. pi05 has about 3B parameters and does not fit on a T4. smolvla alone runs on a T4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version   # needs 3.12+

## 1. Get the code and install

### 1a. Clone the repo

This clones into `/content/lerobot-policy-probe`. If the folder is already there, for example when you re-run the cell, it pulls the latest `main` instead.

In [ ]:
%cd /content
!if [ -d lerobot-policy-probe/.git ]; then git -C lerobot-policy-probe pull --ff-only; \
  else git clone https://github.com/alimertturker/lerobot-policy-probe.git; fi
%cd /content/lerobot-policy-probe
!git log --oneline -3

### 1b. System packages

LeRobot decodes dataset videos with ffmpeg. Colab ships an ffmpeg binary, but this makes sure the development libraries are there too.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg libavcodec-dev libavformat-dev libswscale-dev > /dev/null
!ffmpeg -version | head -1

### 1c. Install LeRobot from source

The probe needs `lerobot>=0.6.2`. That version is not on PyPI yet (PyPI stops at 0.6.1), so this installs it from the LeRobot GitHub repo, together with the extras these checkpoints need:
- `dataset`: reading a `LeRobotDataset` (for `score` and `replay`)
- `smolvla`: transformers, accelerate, and num2words
- `pi`: transformers and scipy, for pi05

`LEROBOT_REF` can be a branch, tag, or commit. Pin a commit if you want identical results across runs. This takes a few minutes.

In [ ]:
LEROBOT_REF = "main"
!pip install -q --upgrade pip
!pip install -q "lerobot[dataset,smolvla,pi] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT_REF}"

### 1d. Install the probe

This goes second, so pip sees that the lerobot requirement is already met and doesn't try to fetch it from PyPI. It's editable, so a `git pull` in 1a takes effect without reinstalling.

In [ ]:
!pip install -q -e ".[plot]"

### 1e. Restart if needed

If pip printed a torch, numpy, or transformers conflict with Colab's preinstalled versions, use **Runtime → Restart session**. Then run the next cell to check the install; you don't need to reinstall.

In [ ]:
%cd /content/lerobot-policy-probe
import importlib.metadata as md
for pkg in ["lerobot", "lerobot-policy-probe", "torch", "transformers", "matplotlib"]:
    try:
        print(f"{pkg:22s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:22s} MISSING")

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
!lerobot-probe --help | head -12

## 2. Settings

All three checkpoints and the dataset are public, so you don't need a Hugging Face token.

In [ ]:
DATASET = "alimerido/wrist-top-cube_20260705_134536"
POLICIES = {
    # label: (policy type, checkpoint)
    "smolvla":  ("smolvla", "alimerido/smolvla-wrist-top-cube-v3"),
    "pi05":     ("pi05",    "alimerido/pi05-wrist-top-generic-3k-tn"),
    "pi05-10k": ("pi05",    "alimerido/pi05-wrist-top-cube-10k"),
}
EPISODE_IDS = [0, 1, 2, 3, 4]   # exact episodes to replay; set to [] to pick EPISODES at random
EPISODES = 3      # only used when EPISODE_IDS is empty
REPLAN = 50       # steps each chunk runs open loop before the next observation
MAX_STEPS = 600   # cap per episode
OUT = "outputs/replay"

## 3. Inspect the weights first (cheap, no GPU)

This is worth a minute before replaying anything. If a component reads **RANDOM**, the policy is blind there, and the replay will only confirm what you already know. It reads a few MB over HTTP instead of downloading the checkpoint.

In [ ]:
!lerobot-probe inspect alimerido/smolvla-wrist-top-cube-v3 --base lerobot/smolvla_base
!lerobot-probe inspect alimerido/pi05-wrist-top-generic-3k-tn --base lerobot/pi05_base
!lerobot-probe inspect alimerido/pi05-wrist-top-cube-10k --base lerobot/pi05_base

## 4. Replay

Each policy runs in its own process, so its GPU memory is freed before the next one loads. Execution matches deployment: the policy sees a real frame every `REPLAN` steps, and the chunk runs open loop in between.

All runs write to the same directory, so the plot overlays them.

In [ ]:
EP_ARGS = f"--episode-ids {' '.join(map(str, EPISODE_IDS))}" if EPISODE_IDS else f"--episodes {EPISODES}"
for label, (ptype, ckpt) in POLICIES.items():
    print(f"\n===== {label} =====")
    !lerobot-probe replay --policy-type {ptype} --checkpoint {ckpt} --dataset {DATASET} \
        {EP_ARGS} --replan {REPLAN} --max-steps {MAX_STEPS} \
        --label {label} --out {OUT} --device cuda

## 5. Plot

In [ ]:
!lerobot-probe plot --traces {OUT}

from pathlib import Path
from IPython.display import Image, display
for png in sorted(Path(OUT).glob("episode_*.png")):
    print(png.name)
    display(Image(str(png)))

### Reading the plots

There is one panel per joint. Solid black is the human, and each policy is a dashed line.

| what you see | what it means |
|---|---|
| tracks the human, with a sawtooth at each replan | healthy |
| offset from the very first sample | not tracking the arm: on hardware it jumps, lunges, then reverses |
| dithers around the human | noisy chunk, so the arm buzzes regardless of latency |
| flat | collapsed to a constant, usually untrained |

## 6. Optional: score against a hold-still baseline

This gives one number per policy. A negative "vs doing nothing" means the policy does worse than repeating the current joint positions.

In [ ]:
for label, (ptype, ckpt) in POLICIES.items():
    print(f"\n===== {label} =====")
    !lerobot-probe score --policy-type {ptype} --checkpoint {ckpt} --dataset {DATASET} --device cuda

## 7. Keep the results

The Colab disk is wiped when the runtime stops. Copy the outputs to Drive:

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/lerobot-probe && cp -r {OUT} /content/drive/MyDrive/lerobot-probe/